In [ ]:
import torch
from pathlib import Path
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import loss
import pandas as pd
import json


dataset = h5py.File("dataset_filtered/train_target.h5", 'r')

In [ ]:

with open("dataset_filtered/center_sizes.json", "r") as f:
    center_sizes = json.load(f)

# Calculate constants

In [ ]:
shifts = {
    "au": 1.01,
    "gdf3": 0.95,
    "laf3": 0.99,
    "tbf3": 0.94
}
scale_factor = 4
calibration_constants = {}
for n, d in dataset.items():
    print(n)
    print(len(d))
    input2d = torch.from_numpy(d[:])[:, None]
    df = pd.read_csv(Path("dataset_filtered") / n , sep=r'\s+')
    target = (df.q.to_numpy(), df.I.to_numpy(), center_sizes[n] * scale_factor)
    with torch.no_grad():
        device = input2d.device
        summed_input2d = loss.sum_aligned_images(input2d)
        # summed_input2d = F.interpolate(summed_input2d.unsqueeze(0), (512, 512), mode="bicubic")
        radial_distance, intensity = loss.calc_radial_distribution(summed_input2d.squeeze())

        q, I, center_size = target
        q, I = torch.from_numpy(q), torch.from_numpy(I)

        # remove center and normalize
        intensity[:center_size] = 0
        intensity = intensity / intensity[:intensity.shape[0] // 2].max()

        max_target = q[I.argmax()]
        max_input = torch.argmax(intensity[:intensity.shape[0] // 2])
        # calibration_constant = max_xrd / max_eld, this is pixels -> q
        # we are going q -> pixels, so it needs to be max_eld / max_xrd
        calibration_constant = max_input / max_target
        calibration_constant = calibration_constant / shifts.get(n, 1)
        target1d = loss.resize_target(q, I, calibration_constant )

        # Save constant for original resolution
        calibration_constants[n] = calibration_constant.item() / scale_factor

        target1d = torch.nn.functional.pad(target1d, (0, intensity.shape[0] - target1d.shape[0]))
    
    plt.imshow(torch.log10_(summed_input2d[0] + 1).cpu())
    plt.show()
    plt.plot(target1d[:300])
    plt.plot(intensity[:300])
    plt.show()

    del input2d

print(json.dumps(calibration_constants, indent=4))

# Verify 2x scale

In [ ]:
scale_factor = 2
for n, d in dataset.items():
    print(n)
    print(len(d))
    input2d = torch.from_numpy(d[:])[:, None]
    df = pd.read_csv(Path("dataset_filtered") / n , sep=r'\s+')
    target = (df.q.to_numpy(), df.I.to_numpy(), center_sizes[n] * scale_factor)
    with torch.no_grad():
        device = input2d.device
        summed_input2d = loss.sum_aligned_images(input2d)
        summed_input2d = F.interpolate(summed_input2d.unsqueeze(0), (512, 512), mode="bicubic")
        summed_input2d = summed_input2d.clamp(0, None)
        radial_distance, intensity = loss.calc_radial_distribution(summed_input2d.squeeze())

        q, I, center_size = target
        q, I = torch.from_numpy(q), torch.from_numpy(I)

        # remove center and normalize
        intensity[:center_size] = 0
        intensity = intensity / intensity[:intensity.shape[0] // 2].max()

        calibration_constant = calibration_constants[n] * scale_factor
        target1d = loss.resize_target(q, I, calibration_constant )

        target1d = torch.nn.functional.pad(target1d, (0, intensity.shape[0] - target1d.shape[0]))
    
    plt.imshow(torch.log10_(summed_input2d[0, 0] + 1).cpu())
    plt.show()
    plt.plot(target1d[:300])
    plt.plot(intensity[:300])
    plt.show()

    del input2d